# UC5: ATT Estimation for Strategy Switching

**Scientific question**: Among campaigns that *actually used* Bayesian optimisation,
what would their outcome have been if they had used random search instead?

**Estimand**: Average Treatment Effect on the Treated (ATT)
$$\text{ATT} = E[Y(1) - Y(0) \mid T = 1]$$

The first term $E[Y(1) \mid T = 1]$ is directly observed (BO campaign outcomes).
The second term $E[Y(0) \mid T = 1]$ is the counterfactual: what would those same
campaigns have achieved under random search?

**Why ATT rather than ATE?**
UC1 estimates the ATE — the average effect across *all* campaigns, including those
where BO was never used. The ATT answers a more actionable retrospective question:
*was the practitioner's choice of BO justified, given the campaigns they ran?*

**Positive selection hypothesis**: Because practitioners preferentially apply BO to
easier campaigns (high SNR, smooth landscape, large budget — the UC1/UC2 confounders),
the treated group is positively selected. In this regime, ATT > ATE: BO's retrospective
value is larger than its population-average value, because it was deployed where it
works best.

**DoWhy feature**: Counterfactual estimation (backdoor identification + IPW for ATT;
interaction OLS for individual counterfactuals)

**Design rule target**: DR-H — when to trust retrospective BO performance claims

**DAG**: UC2 (identical structure; ATT is an estimation-layer change, not a structural one)


In [1]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
import dowhy
from dowhy import CausalModel

from src.graphs import (
    uc5_dag, get_uc5_adjustment_set,
    UC5_SNR_LOW_MAX, UC5_SNR_HIGH_MIN,
    UC2_SNR_THRESHOLD,
)
from src.simulator import run_campaigns

sns.set_theme(style="ticks", font_scale=1.1)
SEED = 42

print("dowhy version:", dowhy.__version__)
print("Adjustment set:", get_uc5_adjustment_set())
print("SNR bins: low < {}, mid [{}, {}), high ≥ {}".format(
    UC5_SNR_LOW_MAX, UC5_SNR_LOW_MAX, UC5_SNR_HIGH_MIN, UC5_SNR_HIGH_MIN))


dowhy version: 0.14
Adjustment set: ['budget', 'dimensionality', 'landscape_smoothness', 'noise_level', 'snr']
SNR bins: low < 1.5, mid [1.5, 3.0), high ≥ 3.0


## 1. Simulate UC5 campaigns (synthetic data)

UC5 uses the UC2 DGP — the SNR-modulated strategy effect is essential for
the positive selection story. `use_case='UC5'` is an alias for `'UC2'` in
the simulator; the ATT estimand is an estimation-layer change only.

In [2]:
# n=2000 gives enough power to estimate ATT to ±0.01.
# All data is synthetic — generated by the UC2 structural causal model.
df = run_campaigns(n_campaigns=2000, seed=SEED, use_case="UC5")

# True ATE and ATT are known by construction from snr_modulation
TRUE_ATE = df["snr_modulation"].mean()
TRUE_ATT = df.loc[df["strategy"] == 1, "snr_modulation"].mean()

print(f"Shape:            {df.shape}")
print(f"BO rate:          {df['strategy'].mean():.3f}  (target ≈ 0.50)")
print(f"Mean SNR:         {df['snr'].mean():.2f}")
print(f"Treated (BO) n:   {df['strategy'].sum()}")
print(f"Control (RS) n:   {(df['strategy']==0).sum()}")
print()
print(f"True ATE  E[snr_mod]:        {TRUE_ATE:+.4f}")
print(f"True ATT  E[snr_mod | T=1]:  {TRUE_ATT:+.4f}")
print(f"ATT > ATE (positive selection): {TRUE_ATT > TRUE_ATE}")
print(f"True gap (ATT - ATE):           {TRUE_ATT - TRUE_ATE:+.4f}  "
      f"({100*(TRUE_ATT-TRUE_ATE)/TRUE_ATE:+.1f}% of ATE)")


Shape:            (2000, 14)
BO rate:          0.495  (target ≈ 0.50)
Mean SNR:         15.26
Treated (BO) n:   990
Control (RS) n:   1010

True ATE  E[snr_mod]:        +0.0828
True ATT  E[snr_mod | T=1]:  +0.0875
ATT > ATE (positive selection): True
True gap (ATT - ATE):           +0.0047  (+5.7% of ATE)


## 2. Positive selection — confounder imbalance

If practitioners preferentially apply BO to easier campaigns, BO campaigns should
have higher SNR, smoother landscapes, and larger budgets than RS campaigns.
This imbalance is the mechanism behind ATT > ATE.


In [3]:
confounders = ["landscape_smoothness", "noise_level", "budget", "dimensionality", "snr"]
bo  = df[df["strategy"] == 1]
rs  = df[df["strategy"] == 0]

print(f"{'Covariate':<25}  {'BO mean':>10}  {'RS mean':>10}  {'Diff':>10}  {'Direction'}")
print("-" * 70)
for c in confounders:
    bo_m = bo[c].mean()
    rs_m = rs[c].mean()
    diff = bo_m - rs_m
    direction = "BO easier" if (c in ["landscape_smoothness", "budget", "snr"] and diff > 0) or \
                               (c in ["noise_level", "dimensionality"] and diff < 0) else "BO harder"
    print(f"{c:<25}  {bo_m:>10.3f}  {rs_m:>10.3f}  {diff:>+10.3f}  {direction}")

print()
print("Positive selection confirmed: BO campaigns have higher SNR, smoother")
print("landscapes, and larger budgets — exactly the conditions where BO excels.")


Covariate                     BO mean     RS mean        Diff  Direction
----------------------------------------------------------------------
landscape_smoothness            0.773       0.666      +0.106  BO easier
noise_level                     0.107       0.142      -0.035  BO easier
budget                         61.438      57.272      +4.166  BO easier
dimensionality                  4.151       4.254      -0.104  BO easier
snr                            19.353      11.245      +8.108  BO easier

Positive selection confirmed: BO campaigns have higher SNR, smoother
landscapes, and larger budgets — exactly the conditions where BO excels.


## 3. DoWhy: identification

The ATT is identified by the same backdoor criterion as the ATE (UC1/UC2).
The DAG is identical to UC2 — no new nodes or edges are needed.


In [4]:
G = uc5_dag()

# DoWhy uses latent_outcome aliased as norm_best_outcome (avoids ceiling clipping)
df_dw = df.copy()
df_dw["norm_best_outcome"] = df_dw["latent_outcome"]

model = CausalModel(
    data=df_dw,
    treatment="strategy",
    outcome="norm_best_outcome",
    graph=G,
)
identified_estimand = model.identify_effect(proceed_when_unidentifiable=False)
backdoor_vars = identified_estimand.get_backdoor_variables()

# NOTE: DoWhy always reports EstimandType.NONPARAMETRIC_ATE here, even when
# target_units="att" is later passed to estimate_effect().  Identification
# is performed for the ATE by default; the ATT restriction is applied at
# *estimation* time (not identification time).  The backdoor adjustment set
# is the same for both ATE and ATT under the no-unmeasured-confounders
# assumption, so the identified estimand is valid for both targets.
print("Estimand type:", identified_estimand.estimand_type,
      "  ← DoWhy labels this ATE; ATT restriction applied at estimation time")
print("Backdoor variables:", sorted(backdoor_vars))
print()
print("Identified estimand:")
print(identified_estimand)


Estimand type: EstimandType.NONPARAMETRIC_ATE
Backdoor variables: ['budget', 'dimensionality', 'landscape_smoothness', 'noise_level', 'snr']

Identified estimand:
Estimand type: EstimandType.NONPARAMETRIC_ATE

### Estimand : 1
Estimand name: backdoor
Estimand expression:
     d                                                                         ↪
───────────(E[norm_best_outcome|landscape_smoothness,snr,noise_level,dimension ↪
d[strategy]                                                                    ↪

↪               
↪ ality,budget])
↪               
Estimand assumption 1, Unconfoundedness: If U→{strategy} and U→norm_best_outcome then P(norm_best_outcome|strategy,landscape_smoothness,snr,noise_level,dimensionality,budget,U) = P(norm_best_outcome|strategy,landscape_smoothness,snr,noise_level,dimensionality,budget)

### Estimand : 2
Estimand name: iv
No such variable(s) found!

### Estimand : 3
Estimand name: frontdoor
No such variable(s) found!

### Estimand : 4
Estimand name:

## 4. ATE estimation (DoWhy backdoor, for comparison with UC1)

We first estimate the ATE using the same DoWhy backdoor linear regression as UC1.
This gives the population-average effect — the baseline for the ATT comparison.


In [5]:
est_ate = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.linear_regression",
    target_units="ate",
    confidence_intervals=True,
    test_significance=True,
)
adj_ate = float(est_ate.value)
ci_ate  = est_ate.get_confidence_intervals()

print(f"Estimated ATE (backdoor LR):  {adj_ate:+.4f}")
print(f"95% CI:                       [{float(ci_ate[0]):+.4f}, {float(ci_ate[1]):+.4f}]")
print(f"True ATE:                     {TRUE_ATE:+.4f}")
print(f"Error:                        {adj_ate - TRUE_ATE:+.4f}")
print()
print("Consistent with UC1: ATE ≈ +0.075 after backdoor adjustment.")


Estimated ATE (backdoor LR):  +0.0749
95% CI:                       [+0.0680, +0.0822]
True ATE:                     +0.0828
Error:                        -0.0080

Consistent with UC1: ATE ≈ +0.075 after backdoor adjustment.


## 5. ATT estimation (AIPW — doubly-robust)

**Why not DoWhy's `target_units='att'` with linear regression?**
Linear regression with additive confounders gives the same coefficient regardless
of `target_units`, because the estimator is global: $\hat{\beta}_1$ is the same
for ATE and ATT when the model is $Y = \beta_0 + \beta_1 T + \beta_2 X + \varepsilon$.
To get ATT ≠ ATE we need an estimator that respects treatment effect heterogeneity.

We use the **Augmented IPW (AIPW)** doubly-robust estimator:

$$\hat{\text{ATT}}_{\text{AIPW}} =
  \underbrace{E[\hat{\mu}_1(X) - \hat{\mu}_0(X) \mid T=1]}_{\text{outcome model}}
  + \underbrace{\frac{E[T(Y - \hat{\mu}_1(X))]}{P(T=1)}}_{\text{treated residual}}
  - \underbrace{\frac{E\left[\frac{(1-T)\hat{e}(X)}{1-\hat{e}(X)}(Y - \hat{\mu}_0(X))\right]}{P(T=1)}}_{\text{control residual}}$$

where $\hat{e}(X) = P(T=1 \mid X)$ is the propensity score and
$\hat{\mu}_t(X) = E[Y \mid T=t, X]$ are the outcome models.
AIPW is consistent if *either* the propensity score *or* the outcome model is
correctly specified ("doubly robust"), and has lower variance than plain IPW.

**Known limitation**: both models are linear approximations to the true sigmoid
SNR effect, so both are slightly misspecified. The AIPW ATT underestimates the
true ATT by ~0.015. This is documented in §10 and the reasoning doc.


In [6]:
# ── Step 1: Fit propensity score model ───────────────────────────────────────
def _std(s): return (s - s.mean()) / (s.std() + 1e-12)

df_dw["ls_s"]  = _std(df_dw["landscape_smoothness"])
df_dw["nl_s"]  = _std(df_dw["noise_level"])
df_dw["bud_s"] = _std(df_dw["budget"])
df_dw["dim_s"] = _std(df_dw["dimensionality"])
df_dw["snr_s"] = _std(df_dw["snr"])

X_ps = df_dw[["ls_s", "nl_s", "bud_s", "dim_s", "snr_s"]].values
T    = df_dw["strategy"].values
Y    = df_dw["norm_best_outcome"].values

lr_ps = LogisticRegression(C=1.0, max_iter=500, random_state=SEED)
lr_ps.fit(X_ps, T)
ps      = lr_ps.predict_proba(X_ps)[:, 1]   # P(T=1 | X)
ps_clip = np.clip(ps, 0.05, 0.95)            # clip for numerical stability

print(f"Propensity score range (clipped): [{ps_clip.min():.3f}, {ps_clip.max():.3f}]")
print(f"Mean PS (treated):  {ps_clip[T==1].mean():.3f}")
print(f"Mean PS (control):  {ps_clip[T==0].mean():.3f}")

# ── Step 2: Outcome models (fit separately on treated / control) ──────────────
# mu1(x) = E[Y | T=1, X=x],  mu0(x) = E[Y | T=0, X=x]
import statsmodels.formula.api as smf
m1 = smf.ols("norm_best_outcome ~ snr_s + ls_s + nl_s + bud_s + dim_s",
             data=df_dw[df_dw["strategy"] == 1]).fit()
m0 = smf.ols("norm_best_outcome ~ snr_s + ls_s + nl_s + bud_s + dim_s",
             data=df_dw[df_dw["strategy"] == 0]).fit()

mu1 = m1.predict(df_dw)   # E[Y(1) | X] for all units
mu0 = m0.predict(df_dw)   # E[Y(0) | X] for all units

# ── Step 3: AIPW ATT estimator (doubly-robust) ───────────────────────────────
# ATT_AIPW = E[mu1(X) - mu0(X) | T=1]
#           + E[T*(Y - mu1(X))] / P(T=1)              ← treated residual
#           - E[(1-T)*ps*(Y - mu0(X)) / (1-ps)] / P(T=1)  ← control residual
#
# "Doubly-robust": consistent if EITHER the propensity score model OR the
# outcome model is correctly specified. Both are linear approximations to
# the true (sigmoid) SNR effect, so both are slightly misspecified — the
# AIPW estimate therefore underestimates the true ATT by ~0.015.
# This is documented as a known limitation (see §10 and uc5_dag_reasoning.md).
p_t = T.mean()
aipw_att = (
    np.mean((mu1 - mu0)[T == 1])
    + np.mean(T * (Y - mu1) / p_t)
    - np.mean((1 - T) * ps_clip * (Y - mu0) / ((1 - ps_clip) * p_t))
)

# AIPW ATE (for comparison)
aipw_ate = (
    np.mean(mu1 - mu0)
    + np.mean(T * (Y - mu1) / ps_clip)
    - np.mean((1 - T) * (Y - mu0) / (1 - ps_clip))
)

# ── Step 4: Bootstrap 95% CI (percentile method, B=2000) ─────────────────────
rng_boot = np.random.default_rng(SEED)
boot_atts = []
n = len(T)
for _ in range(2000):
    idx = rng_boot.integers(0, n, n)
    T_b, Y_b, ps_b = T[idx], Y[idx], ps_clip[idx]
    mu1_b, mu0_b   = mu1[idx], mu0[idx]
    p_t_b = T_b.mean()
    if p_t_b == 0:
        continue
    att_b = (
        np.mean((mu1_b - mu0_b)[T_b == 1])
        + np.mean(T_b * (Y_b - mu1_b) / p_t_b)
        - np.mean((1 - T_b) * ps_b * (Y_b - mu0_b) / ((1 - ps_b) * p_t_b))
    )
    boot_atts.append(att_b)
ci_lo, ci_hi = np.percentile(boot_atts, [2.5, 97.5])

print(f"\nEstimated ATT (AIPW):  {aipw_att:+.4f}")
print(f"Bootstrap 95% CI:      [{ci_lo:+.4f}, {ci_hi:+.4f}]")
print(f"True ATT:              {TRUE_ATT:+.4f}")
print(f"Error:                 {aipw_att - TRUE_ATT:+.4f}")
print()
print(f"AIPW ATE:              {aipw_ate:+.4f}  (true: {TRUE_ATE:+.4f})")
print(f"ATT > ATE:             {aipw_att > aipw_ate}")
print(f"ATT - ATE:             {aipw_att - aipw_ate:+.4f}")
print()
print("Note: both AIPW models are linear approximations to the true sigmoid")
print("SNR effect, so both slightly underestimate the true ATT (+0.088).")
print("The ATT > ATE ordering is robust across all estimators that allow")
print("treatment effect heterogeneity (see uc5_dag_reasoning.md §6).")


Propensity score range (clipped): [0.164, 0.950]
Mean PS (treated):  0.523
Mean PS (control):  0.467



Estimated ATT (AIPW):  +0.0725
Bootstrap 95% CI:      [+0.0604, +0.0846]
True ATT:              +0.0875
Error:                 -0.0150

AIPW ATE:              +0.0687  (true: +0.0828)
ATT > ATE:             True
ATT - ATE:             +0.0038

Note: both AIPW models are linear approximations to the true sigmoid
SNR effect, so both slightly underestimate the true ATT (+0.088).
The ATT > ATE ordering is robust across all estimators that allow
treatment effect heterogeneity (see uc5_dag_reasoning.md §6).


## 6. Individual counterfactual distributions

For each BO-treated campaign, we predict the counterfactual outcome $\hat{Y}(0)$
using an OLS model with a `strategy × snr` interaction term. This captures the
SNR-dependent treatment effect heterogeneity identified in UC2.

The per-campaign switching loss $Y(1) - \hat{Y}(0)$ shows the distribution of
BO's benefit: most campaigns benefit, but ~32% would have done equally well or
better under random search (typically low-SNR campaigns).


In [7]:
# ── Interaction OLS model for individual counterfactuals ─────────────────────
# strategy × snr_s captures the SNR-dependent BO advantage (UC2 DR-B).
#
# IMPORTANT: The interaction term (strategy:snr_s) has p ≈ 0.057, which is
# marginally non-significant at the conventional α = 0.05 threshold.
# This means the *estimated* heterogeneity (ATT_high > ATT_low) is statistically
# weak in this single dataset.  The true heterogeneity is real by DGP construction
# (snr_modulation is a function of SNR), but the estimated stratified ATTs carry
# substantial uncertainty — especially in the low-SNR bin (n=38 treated).
# Interpret stratified ATT estimates as directional, not precise.
m_interact = smf.ols(
    "norm_best_outcome ~ strategy * snr_s + ls_s + nl_s + bud_s + dim_s",
    data=df_dw
).fit()

print("Interaction model coefficients:")
for name in ["strategy", "snr_s", "strategy:snr_s"]:
    print(f"  {name:<20s}: {m_interact.params[name]:+.4f}  "
          f"(p={m_interact.pvalues[name]:.3f})")

# Predict Y(0) for each treated campaign
treated = df_dw[df_dw["strategy"] == 1].copy()
treated_cf = treated.copy()
treated_cf["strategy"] = 0

y1_obs  = treated["norm_best_outcome"].values
y0_pred = m_interact.predict(treated_cf).values
cf_loss = y1_obs - y0_pred

print(f"\nIndividual switching loss  Y(1) − Ŷ(0)  (n={len(cf_loss)} BO campaigns):")
print(f"  Mean:    {cf_loss.mean():+.4f}  (≈ ATT from interaction model)")
print(f"  Median:  {np.median(cf_loss):+.4f}")
print(f"  Std:     {cf_loss.std():.4f}")
print(f"  % BO helped (loss > 0): {(cf_loss > 0).mean()*100:.1f}%")
print(f"  % BO hurt  (loss < 0): {(cf_loss < 0).mean()*100:.1f}%")
print(f"  % large benefit (> 0.10): {(cf_loss > 0.10).mean()*100:.1f}%")


Interaction model coefficients:
  strategy            : +0.0700  (p=0.000)
  snr_s               : +0.0673  (p=0.000)
  strategy:snr_s      : +0.0159  (p=0.057)

Individual switching loss  Y(1) − Ŷ(0)  (n=990 BO campaigns):
  Mean:    +0.0729  (≈ ATT from interaction model)
  Median:  +0.0660
  Std:     0.1298
  % BO helped (loss > 0): 68.2%
  % BO hurt  (loss < 0): 31.8%
  % large benefit (> 0.10): 42.4%


## 7. SNR-stratified ATT

UC2 showed that BO's advantage is SNR-dependent (DR-B: advantage collapses below
τ = 2.0). The ATT should therefore be largest for high-SNR BO campaigns and
smallest (or negative) for low-SNR BO campaigns.

SNR bins straddle the UC2 threshold τ = 2.0:
- **Low**: SNR < 1.5 (below threshold — BO surrogate unreliable)
- **Mid**: 1.5 ≤ SNR < 3.0 (near threshold — mixed regime)
- **High**: SNR ≥ 3.0 (above threshold — BO surrogate reliable)


In [8]:
bins = [
    ("low",  df_dw["snr"] < UC5_SNR_LOW_MAX),
    ("mid",  (df_dw["snr"] >= UC5_SNR_LOW_MAX) & (df_dw["snr"] < UC5_SNR_HIGH_MIN)),
    ("high", df_dw["snr"] >= UC5_SNR_HIGH_MIN),
]

print(f"{'Bin':6s}  {'n_total':>8}  {'n_treated':>10}  {'mean_SNR':>9}  "
      f"{'true_ATT':>9}  {'est_ATT':>9}")
print("-" * 65)

strat_results = {}
for name, mask in bins:
    treated_bin    = df_dw[mask & (df_dw["strategy"] == 1)].copy()
    treated_bin_cf = treated_bin.copy()
    treated_bin_cf["strategy"] = 0

    y1_bin = treated_bin["norm_best_outcome"].values
    y0_bin = m_interact.predict(treated_bin_cf).values
    est_att_bin  = (y1_bin - y0_bin).mean()
    true_att_bin = df.loc[mask & (df["strategy"] == 1), "snr_modulation"].mean()

    strat_results[name] = {
        "n_total":   int(mask.sum()),
        "n_treated": len(treated_bin),
        "mean_snr":  treated_bin["snr"].mean(),
        "true_att":  true_att_bin,
        "est_att":   est_att_bin,
    }
    print(f"{name:6s}  {mask.sum():>8}  {len(treated_bin):>10}  "
          f"{treated_bin['snr'].mean():>9.2f}  "
          f"{true_att_bin:>+9.4f}  {est_att_bin:>+9.4f}")

print()
ordering_true = (strat_results["low"]["true_att"] < strat_results["mid"]["true_att"]
                 < strat_results["high"]["true_att"])
ordering_est  = (strat_results["low"]["est_att"]  < strat_results["mid"]["est_att"]
                 < strat_results["high"]["est_att"])
print(f"Ordering (true):      low < mid < high? {ordering_true}")
print(f"Ordering (estimated): low < mid < high? {ordering_est}")
print()
print("Note: low-SNR bin has only", strat_results["low"]["n_treated"],
      "treated campaigns — BO is rarely used at low SNR (positive selection).")
print("True ATT is negative at low SNR: BO slightly hurts when the surrogate is unreliable.")
print()
print("WARNING (sign error in low-SNR estimate):")
print(f"  True ATT_low  = {strat_results['low']['true_att']:+.4f}  (negative — BO hurts)")
print(f"  Est. ATT_low  = {strat_results['low']['est_att']:+.4f}  (positive — WRONG SIGN)")
print("  The interaction OLS model extrapolates linearly into the low-SNR region")
print("  where the true effect is non-linear and negative.  With only",
      strat_results["low"]["n_treated"], "treated units, the low-SNR")
print("  bin is too underpowered to estimate the sign reliably.")
print("  Do NOT use the estimated ATT_low as a point estimate; use the true ATT_low")
print("  (known by DGP construction) to confirm the direction.")


Bin      n_total   n_treated   mean_SNR   true_ATT    est_ATT
-----------------------------------------------------------------
low          111          38       1.07    -0.0067    +0.0165
mid          238          87       2.29    +0.0353    +0.0307
high        1651         865      21.87    +0.0969    +0.0797

Ordering (true):      low < mid < high? True
Ordering (estimated): low < mid < high? True

Note: low-SNR bin has only 38 treated campaigns — BO is rarely used at low SNR (positive selection).
True ATT is negative at low SNR: BO slightly hurts when the surrogate is unreliable.


## 8. Summary figure

Three panels:
- **A**: Observed Y(1) vs counterfactual Y(0) for each BO campaign
- **B**: Distribution of individual switching losses with ATT and ATE reference lines
- **C**: SNR-stratified ATT showing the noise-regime dependence


In [9]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# ── Panel A: Y(1) vs Y(0) scatter ────────────────────────────────────────────
ax = axes[0]
ax.scatter(y0_pred, y1_obs, alpha=0.25, s=18, color="#0279EE", rasterized=True)
lims = [min(y0_pred.min(), y1_obs.min()) - 0.02,
        max(y0_pred.max(), y1_obs.max()) + 0.02]
ax.plot(lims, lims, "k--", linewidth=1.2, label="Y(1) = Y(0) (no effect)")
ax.set_xlabel("Counterfactual Y(0)\n(predicted outcome under random search)", fontsize=9)
ax.set_ylabel("Observed Y(1)\n(actual outcome under BO)", fontsize=9)
ax.set_title(f"A. Observed vs counterfactual\n(BO-treated campaigns, n={len(y1_obs)})", fontsize=10)
ax.legend(fontsize=8)
sns.despine(ax=ax)

# ── Panel B: Distribution of switching losses ─────────────────────────────────
ax = axes[1]
ax.hist(cf_loss, bins=40, color="#FF9400", alpha=0.75, edgecolor="white", linewidth=0.4)
ax.axvline(0,         color="#B0B0B0", linewidth=1.0, linestyle=":")
ax.axvline(aipw_att,  color="#FF9400", linewidth=2.2, linestyle="-",
           label=f"ATT (AIPW) = {aipw_att:+.3f}")
ax.axvline(aipw_ate,  color="#0279EE", linewidth=1.8, linestyle="--",
           label=f"ATE (AIPW) = {aipw_ate:+.3f}")
ax.set_xlabel("Switching loss  Y(1) − Ŷ(0)", fontsize=10)
ax.set_ylabel("Count", fontsize=10)
ax.set_title("B. Individual switching losses\n(BO → random search counterfactual)", fontsize=10)
ax.legend(fontsize=8)
pct_pos = (cf_loss > 0).mean() * 100
ax.text(0.97, 0.97, f"{pct_pos:.0f}% BO helped\n{100-pct_pos:.0f}% BO hurt",
        transform=ax.transAxes, ha="right", va="top", fontsize=8.5,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
sns.despine(ax=ax)

# ── Panel C: SNR-stratified ATT ───────────────────────────────────────────────
ax = axes[2]
bin_names = ["Low\n(SNR < 1.5)", "Mid\n(1.5 ≤ SNR < 3)", "High\n(SNR ≥ 3)"]
true_atts = [strat_results[k]["true_att"] for k in ["low", "mid", "high"]]
est_atts  = [strat_results[k]["est_att"]  for k in ["low", "mid", "high"]]
n_treated = [strat_results[k]["n_treated"] for k in ["low", "mid", "high"]]

x = np.arange(3)
w = 0.35
ax.bar(x - w/2, true_atts, w, color="#ECE9E2", edgecolor="#555",
       linewidth=1.2, label="True ATT")
ax.bar(x + w/2, est_atts,  w, color="#75A025", edgecolor="#555",
       linewidth=1.2, label="Estimated ATT (OLS)")
ax.axhline(0,         color="#B0B0B0", linewidth=0.8, linestyle=":")
ax.axhline(aipw_ate,  color="#0279EE", linewidth=1.5, linestyle="--",
           label=f"ATE = {aipw_ate:+.3f}")
ax.axhline(aipw_att,  color="#FF9400", linewidth=1.5, linestyle="-",
           label=f"ATT = {aipw_att:+.3f}")
for xi, n in zip(x, n_treated):
    ymin = min(true_atts[xi], est_atts[xi]) - 0.007
    ax.text(xi, ymin, f"n={n}", ha="center", va="top", fontsize=8, color="#555")
ax.set_xticks(x)
ax.set_xticklabels(bin_names, fontsize=9)
ax.set_ylabel("ATT estimate", fontsize=10)
ax.set_title("C. SNR-stratified ATT\n(BO advantage by noise regime)", fontsize=10)
ax.legend(fontsize=7.5, loc="upper left")
sns.despine(ax=ax)

fig.suptitle(
    "UC5: ATT Estimation — What Would BO Campaigns Have Achieved Under Random Search?\n"
    f"ATT (AIPW) = {aipw_att:+.3f}  >  ATE (AIPW) = {aipw_ate:+.3f}  "
    f"(positive selection: BO deployed on easier campaigns)",
    fontsize=10, y=1.02
)
plt.tight_layout()
plt.savefig("../uc5_att_counterfactuals.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: uc5_att_counterfactuals.png")


Saved: uc5_att_counterfactuals.png


## 9. Refutation tests

Three standard DoWhy refutation tests applied to the backdoor estimate.

**What these tests validate**: DoWhy refutation tests probe the *identification
assumptions* — specifically unconfoundedness (no unmeasured common causes) and
the absence of spurious associations.  These assumptions are shared by both the
ATE and ATT estimands, so passing these tests supports the validity of the
backdoor adjustment for both targets.

**What these tests do NOT validate**: The refutation uses DoWhy's backdoor
linear regression with `target_units='att'`.  Because a global linear model
gives ATT = ATE by construction, the refuted estimate (+0.0749) is the ATE,
not the AIPW ATT (+0.0725) reported in §5.  The refutation tests therefore
do **not** directly validate the AIPW point estimate or its confidence interval.
They validate the underlying causal identification, which is a prerequisite
for any ATT estimator.


In [10]:
# DoWhy refutation uses the backdoor LR estimate (target_units='att')
est_att_dw = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.linear_regression",
    target_units="att",
    confidence_intervals=True,
    test_significance=True,
)

ref_rcc = model.refute_estimate(
    identified_estimand, est_att_dw,
    method_name="random_common_cause", random_seed=SEED)

ref_placebo = model.refute_estimate(
    identified_estimand, est_att_dw,
    method_name="placebo_treatment_refuter",
    placebo_type="permute", random_seed=SEED)

ref_subset = model.refute_estimate(
    identified_estimand, est_att_dw,
    method_name="data_subset_refuter",
    subset_fraction=0.8, random_seed=SEED)

print(f"{'Test':<28} {'New effect':>12} {'p-value':>9} {'Pass':>6}")
print("-" * 60)
for name, ref in [("Random common cause",  ref_rcc),
                  ("Placebo treatment",    ref_placebo),
                  ("Data subset (80%)",    ref_subset)]:
    p      = ref.refutation_result['p_value']
    passed = p > 0.05
    print(f"{name:<28} {ref.new_effect:>+12.4f} {p:>9.3f} {'YES' if passed else 'NO':>6}")

all_passed = all(r.refutation_result['p_value'] > 0.05
                 for r in [ref_rcc, ref_placebo, ref_subset])
print(f"\nAll binary tests passed: {all_passed}")


Test                           New effect   p-value   Pass
------------------------------------------------------------
Random common cause               +0.0749     0.980    YES
Placebo treatment                 +0.0002     1.000    YES
Data subset (80%)                 +0.0749     0.940    YES

All binary tests passed: True


## 10. Design rule DR-H

> **DR-H (Retrospective BO value)**: Among campaigns where BO was used, switching
> to random search would have reduced the best outcome by **ATT ≈ +0.073** normalised
> units on average (AIPW, 95% CI: [+0.060, +0.085]; true ATT = +0.088).
> The ATT exceeds the population ATE (+0.069 AIPW) because practitioners deploy BO
> preferentially on high-SNR campaigns where its surrogate model is most reliable.
>
> **Stratified rule**: The switching loss is concentrated in high-SNR campaigns
> (ATT_high ≈ +0.080, n=865). At low SNR (< 1.5), the *true* ATT is negative
> (true ATT_low ≈ −0.007, n=38 treated), meaning BO slightly hurts when the
> surrogate is unreliable. **Caveat**: the *estimated* ATT_low is +0.017 (wrong
> sign), because the interaction OLS model extrapolates linearly into a region
> where the true effect is non-linear and the bin is severely underpowered
> (n=38). The stratified heterogeneity claim rests on the true DGP pattern and
> on the interaction term (p=0.057, marginally non-significant at α=0.05);
> treat it as directional evidence, not a precise estimate.
>
> **Practical implication**: Retrospective performance comparisons that report
> "BO outperformed random search by X" are measuring the ATT, not the ATE.
> The ATT overstates BO's benefit for a *new* campaign drawn from the full
> landscape distribution. To estimate the benefit for a new campaign, use the
> ATE and condition on the campaign's SNR using DR-B.

### Summary table

| Quantity | Estimated | True | Estimator | Note |
|---|---|---|---|---|
| ATE | +0.069 | +0.083 | AIPW | — |
| ATT | +0.073 | +0.088 | AIPW (doubly-robust) | — |
| ATT − ATE | +0.004 | +0.005 | — | — |
| ATT (high SNR) | +0.080 | +0.097 | Interaction OLS | directional |
| ATT (mid SNR)  | +0.031 | +0.035 | Interaction OLS | directional |
| ATT (low SNR)  | +0.017 | −0.007 | Interaction OLS | **sign error** — n=38, underpowered |

**Estimator note**: Both the propensity score model (logistic regression) and the
outcome models (OLS) are linear approximations to the true sigmoid SNR effect.
Both are therefore slightly misspecified, causing AIPW to underestimate the true
ATT by ~0.015. The ATT > ATE ordering is robust across all estimators that allow
treatment effect heterogeneity (interaction OLS, Hajek IPW, AIPW).

**Heterogeneity caveat**: The interaction term (strategy:snr_s, p=0.057) is
marginally non-significant at α=0.05. The stratified ATT ordering
(ATT_high > ATT_mid > ATT_low) is confirmed by the true DGP values but is
statistically weak in the estimated values. The low-SNR estimated ATT has the
wrong sign vs the true value. Treat stratified estimates as directional.
